# Pairsense Animation Enhancer (Real-ESRGAN)

Follow these steps to enhance your animation frames using free Google Colab GPUs:

1. Go to **Runtime > Change runtime type** and ensure **Hardware accelerator** is set to **T4 GPU**.
2. Run the **Setup** cell below by clicking the play button on the left.
3. In the left sidebar of this screen, click the **Folder icon** to open the Files panel.
4. Upload the `animations_to_enhance.zip` file that has been generated in your local project root.
5. Run the **Unzip & Enhance** cell below. It will automatically find the zip, extract it, and process all your images!

In [ ]:
# 1. Setup Real-ESRGAN (Run this cell first)
!git clone https://github.com/xinntao/Real-ESRGAN.git
%cd Real-ESRGAN
!pip install basicsr facexlib gfpgan
!pip install -r requirements.txt
!python setup.py develop

# Fix for torchvision compatibility issue in basicsr
import os, site
for p in site.getsitepackages():
    degradations_path = os.path.join(p, 'basicsr', 'data', 'degradations.py')
    if os.path.exists(degradations_path):
        with open(degradations_path, 'r') as f:
            content = f.read()
        content = content.replace('from torchvision.transforms.functional_tensor import rgb_to_grayscale', 'from torchvision.transforms.functional import rgb_to_grayscale')
        with open(degradations_path, 'w') as f:
            f.write(content)
        break
print("\n✅ Setup complete and basicsr patched!")

In [ ]:
# 2. Unzip & Enhance Images
import os
import glob

%cd /content

# Find the zip file (in case it was uploaded to a subfolder)
zip_path = 'animations_to_enhance.zip'
if not os.path.exists(zip_path):
    found = glob.glob('**/animations_to_enhance.zip', recursive=True)
    if found:
        zip_path = found[0]
        print(f"\n📁 Found zip file at: {zip_path}")

if not os.path.exists(zip_path):
    print("\n⚠️ ERROR: animations_to_enhance.zip was NOT FOUND.")
    print("Please make sure you have fully uploaded the file to the Files panel on the left.")
else:
    print("\n📦 Unzipping files...")
    !unzip -o -q "{zip_path}" -d /content/inputs
    print("✅ Unzip complete.\n")
    
    %cd /content/Real-ESRGAN
    
    input_base = '/content/inputs/public'
    output_base = '/content/outputs'
    
    if not os.path.exists(input_base):
        input_base = '/content/inputs' # Fallback if zip didn't wrap in 'public' directory
        
    print("🚀 Starting enhancement process...")
    # Check if we have valid folders to process
    folders = [f for f in os.listdir(input_base) if os.path.isdir(os.path.join(input_base, f))]
    
    if not folders:
        print(f"⚠️ WARNING: No folders found in {input_base}. The zip might have a different structure.")
        # Fallback to processing the input_base directly if no subfolders exist
        print(f"\n⏳ Enhancing frames in root directory...")
        !python inference_realesrgan.py -n RealESRGAN_x4plus -i "{input_base}" -o "{output_base}" --outscale 2 --ext jpg
        print("✅ Finished root directory")
    else:
        for folder_name in folders:
            input_dir = os.path.join(input_base, folder_name)
            output_dir = os.path.join(output_base, folder_name)
            print(f"\n⏳ Enhancing frames in {folder_name}...")
            !python inference_realesrgan.py -n RealESRGAN_x4plus -i "{input_dir}" -o "{output_dir}" --outscale 2 --ext jpg
            print(f"✅ Finished {folder_name}")
            
    print("\n🎉 All processing complete!")

In [ ]:
# 3. Zip results and download back to your machine
%cd /content
!zip -r enhanced_animations_ready.zip outputs
from google.colab import files
files.download('enhanced_animations_ready.zip')